In [1]:
from dotenv import load_dotenv
import os
load_dotenv("za.env")
from hera.workflows import models, CronWorkflow, Workflow, script, Artifact, Parameter, DAG, Steps, Step, NoneArchiveStrategy
from hera.shared import global_config

global_config.host = "https://dev.services.eodc.eu/workflows/"
global_config.namespace = "spartacus"
global_config.token = os.getenv("ARGO_TOKEN")
global_config.image = "ghcr.io/katharinastarzer21/image_zarr:latest"

security_context = {"runAsUser": 74268,
                    "runAsGroup": 71473}

nfs_volume = [models.Volume(
    name="eodc-mount",
    persistent_volume_claim={"claimName": "eodc-nfs-claim"},
    )]


In [2]:
@script(volume_mounts=[models.VolumeMount(name="eodc-mount", mount_path="/eodc")])

def prepare_zarr(store_path: str = "/eodc/private/tempearth/SPARTACUS.zarr"):
    import datetime
    import numpy as np
    import zarr
 
    now = datetime.datetime.now()
    now_np = np.datetime64(now).astype('datetime64[D]')
    origin = np.datetime64("1961-01-01").astype("datetime64[D]")
 
    new_shape = int((now_np-origin).astype(int))
    new_extent = np.arange(0,new_shape,1)
 
    store = zarr.storage.LocalStore(store_path)
    group = zarr.group(store=store)
 
    array_names=set(group.array_keys())
    coords = {"time", "x", "y"}
    data_arrays = array_names-coords
 
    group["time"].resize(new_shape)
    for array in data_arrays:
        group_shape  = group[array].shape
        group[array].resize((new_shape, group_shape[1], group_shape[2]))
 
    zarr.consolidate_metadata(store)
    store = zarr.storage.LocalStore(store_path)
    group = zarr.group(store=store)
 
    group["time"][:]=new_extent


@script(outputs=Artifact(name="spartacus-file", path="/tmp/output.nc", archive=NoneArchiveStrategy()))

def spartacus_download(variable: str):
    from urllib.request import urlretrieve
    import datetime

    today = datetime.date.today() - datetime.timedelta(days=3)
    year = today.strftime('%Y')
    #year = 2024
    url = f"https://public.hub.geosphere.at/datahub/resources/spartacus-v2-1d-1km/filelisting/{variable}/SPARTACUS2-DAILY_{variable}_{year}.nc"
    urlretrieve(url, "/tmp/output.nc")


@script(
    inputs=Artifact(name="spartacus-file", path="/tmp/output.nc"),
    volume_mounts=[models.VolumeMount(name="eodc-mount", mount_path="/eodc")]
)

def spartacus_write(variable: str):
    import xarray as xr
    import numpy as np
    import zarr
    import datetime

    fill_value = -999
    artifact_path = "/tmp/output.nc" 

    today = datetime.date.today() - datetime.timedelta(days=3)
    year = today.strftime('%Y')

    ds_nc = xr.open_dataset(artifact_path, mask_and_scale=False).load()
    zarr_path = "/eodc/private/tempearth/SPARTACUS.zarr"

    store = zarr.storage.LocalStore(zarr_path)
    group = zarr.group(store=store)
    zarr.consolidate_metadata(store)

    data = ds_nc[variable].values.astype("float32")
    data[data == fill_value] = np.nan
    data_filled = np.where(np.isnan(data), fill_value, data).astype("int16")

    ref_date = np.datetime64("1961-01-01")
    start_date = np.datetime64(f"{year}-01-01")
    start_idx = int((start_date - ref_date) / np.timedelta64(1, "D"))
    num_days = data.shape[0]
    end_idx = start_idx + num_days

    group[variable][start_idx:end_idx, :, :] = data_filled

    
items = ["TX", "TN", "RR", "SA"]

with CronWorkflow(
    generate_name="spartacus-zarr-",
    schedule="17 5 * * *",
    volumes = nfs_volume,
    security_context=security_context,
    entrypoint="workflow"
) as w:
    with DAG(name="pipeline",  inputs=[Parameter(name="item")]) as pipeline:

        download = spartacus_download(arguments={"variable":"{{inputs.parameters.item}}"},
                                 )

        process = spartacus_write(arguments=[{"variable": "{{inputs.parameters.item}}"}, 
                                        download.get_artifact("spartacus-file").with_name("spartacus-file")],
                                        )

        download >> process

    with Steps(name="workflow"):
        prepare_zarr()
        Step(name="parallel-pipelines", template=pipeline, with_param=items, arguments={"item":"{{item}}"})

In [3]:
with open("hera_workflow_spartacus.yaml", "w") as f:
    f.write(w.to_yaml())

In [4]:
w.create()

CronWorkflow(api_version=None, kind=None, metadata=ObjectMeta(annotations=None, cluster_name=None, creation_timestamp=Time(__root__=datetime.datetime(2025, 7, 21, 5, 16, 30, tzinfo=datetime.timezone.utc)), deletion_grace_period_seconds=None, deletion_timestamp=None, finalizers=None, generate_name='spartacus-zarr-', generation=1, labels={'workflows.argoproj.io/creator': 'system-serviceaccount-default-jenkins'}, managed_fields=[ManagedFieldsEntry(api_version='argoproj.io/v1alpha1', fields_type='FieldsV1', fields_v1=FieldsV1(), manager='argo', operation='Update', subresource=None, time=Time(__root__=datetime.datetime(2025, 7, 21, 5, 16, 30, tzinfo=datetime.timezone.utc)))], name='spartacus-zarr-bhvbd', namespace='spartacus', owner_references=None, resource_version='313420191', self_link=None, uid='dc377601-ed9d-4021-a462-b087c06ab539'), spec=CronWorkflowSpec(concurrency_policy=None, failed_jobs_history_limit=None, schedule='17 5 * * *', starting_deadline_seconds=None, successful_jobs_hist